## Usernames filter

This notebook aims to filter the raw usernames contained in `usernames.txt`, to keep only those created between March, 17th and September, 17th, and who let public their production.

### Functions 

To save usernames in `covid_users`

In [ ]:
def save_on_file(X:int, to_save:list, username:str):
    """
    Prints current filtered usernames at the end of the covid_users file.
    """
    with open(f'covid_users_{X}.txt', 'a', encoding='utf-8') as f:
        for user in to_save:
            f.write(user)
        f.write(f'Last: {username}')


To load from files

In [ ]:
def load_batch(X:int)-> tuple[list, bool]:
    """
    Loads remaining part of the usernames to scrap based on last filters.
    Returns a batch of size at most 1,000 userames and a dummy that the batch length is over 1,000.
    """
    with open('usernames.txt', 'r', encoding='utf-8') as f:
        raw_names = f.readlines()

    breakpoint = int(len(raw_names)/3)
    batch = raw_names[breakpoint*(X-1):breakpoint*(X)]
    if X == 3:
        batch += raw_names[breakpoint*(X)+1:]

    # Open already scraped names to avoid rescraping them
    try : 
        with open(f'covid_users_{X}.txt', 'r', encoding='utf-8') as f:
            last_done = f.readlines()[-1]
            if "Last" in last_done:
                last_done = last_done.split(": ", 1)[1]
                batch = batch[batch.index(last_done)+1 : ]
            else : 
                print('No past attempt saved.')
    except Exception:
        print('No previous file found.')

    print(f'Remaining length batch: {len(batch)}')

    if len(batch) >=1000:
        return batch[:999], True
    else : 
        return batch, False

### Chose subset of the total `usernames.txt`

In [1]:
X = 3 # Or 2 or 3

### Scrap Reddit to filter each username in the batch

In [ ]:
import requests
from datetime import datetime
import time
import random 
import string
import numpy as np

start = datetime(2020, 3, 17)
end = datetime(2020, 9, 17)

uncomplete = True


while uncomplete:

    to_save = []
    missing = 0
    headuser = "Mozilla/5.0 (compatible; scraper/1.0)"

    batch, over_1000 = load_batch(X)
    for i, username in enumerate(batch):
        if i % 30 ==0:
            time.sleep(15)
            headuser += random.choice(string.ascii_uppercase + string.digits)
            headers = {
                "User-Agent": headuser
            }
            session = requests.Session()
            session.headers.update(headers)
        if i % 20==0:
            print(f'Step {i}')
        time.sleep(np.random.uniform(low=0.35, high=0.65))
        try:
            r = session.get(
                f"https://www.reddit.com/user/{username.strip('\n')}/about.json", 
                timeout=3)
            r.raise_for_status()
            if str(r.status_code) in ["429", "403", "401"]:
                print("Blocked:", r.status_code)
                raise Exception("Hard block")

            if "application/json" not in r.headers.get("Content-Type", ""):
                print("Soft block (not JSON)")
                print(r.text[:200])
                raise Exception("Soft block")

            created_utc = r.json()["data"]["created_utc"]
        except Exception:
            missing+=1
            if missing % 5 == 0: 
                percent = (missing * 100) / (i + 1)
                print(f"Missing: {percent:.2f}%")
            continue
        date_regis = datetime.utcfromtimestamp(created_utc)
        if start <= date_regis <= end:
            try : 
                r = session.get(
                    f"https://www.reddit.com/user/{username.strip('\n')}/.json",
                    timeout=3)
                r.raise_for_status()
            except Exception : 
                r = session.get(
                    f"https://www.reddit.com/user/{username.strip('\n')}/.json",
                    timeout=20)
                r.raise_for_status()
            data = r.json()
            if data['data']['children']:
                to_save.append(username)
                print(f'Found: {len(to_save)}, Among: {i+1}')

    save_on_file(X, to_save, username)
    if not over_1000:
        uncomplete=False


In [2]:
with open(f'covid_users_{X}.txt', 'r', encoding='utf-8') as f:
    raw_lines = f.readlines()

clean_lines = []
for elt in raw_lines:
    if not elt.startswith('Last:'):
        clean_lines.append(elt)

with open(f'covid_users_{X}.txt', 'w', encoding='utf-8') as f:
    for elt in clean_lines:
        f.write(elt)